In [1]:
import pandas as pd
import numpy as np
import psycopg2
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_recall_curve, auc, classification_report

print("--- Establishing SQL Database Connection Pipeline ---")
try:
    # 1. Connect to PostgreSQL (Update with your actual pgAdmin password)
    conn = psycopg2.connect(
        database="protea_credit_risk",
        user="postgres",
        password="kairu", 
        host="127.0.0.1",
        port="5432"
    )
    
    # 2. Extract our structured data via SQL query
    df = pd.read_sql_query("SELECT * FROM staging_credit_risk;", conn)
    print(f"Data pipeline connected! Loaded {df.shape[0]} loan records.")
    
except Exception as e:
    print(f"Database connection failed: {e}")
finally:
    if 'conn' in locals():
        conn.close()

# --- STEP 2: DATA PRE-PROCESSING & CLEANING ---
# Handle empty cells by dropping missing interest rates or employment lengths
df = df.dropna()

# Split features (X) and target flag (y)
X = df.drop(columns=['loan_status'])
y = df['loan_status']

# One-Hot Encode categorical elements to prepare them for mathematical models
X = pd.get_dummies(X, columns=['person_home_ownership', 'loan_intent', 'loan_grade', 'cb_person_default_on_file'], drop_first=True)

# 80/20 Stratified Train/Test Split (Preserves the exact percentage of defaults in both sets)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# --- STEP 3: TRAIN THE BASELINE MODEL (Logistic Regression) ---
print("\nTraining Baseline Logistic Regression Model...")
baseline = LogisticRegression(max_iter=5000, random_state=42)
baseline.fit(X_train, y_train)

# Calculate Baseline PR-AUC success measure
y_prob_base = baseline.predict_proba(X_test)[:, 1]
precision_b, recall_b, _ = precision_recall_curve(y_test, y_prob_base)
baseline_pr_auc = auc(recall_b, precision_b)

# --- STEP 4: TRAIN THE IMPROVED MODEL (Random Forest Ensemble) ---
print("Training Improved Random Forest Ensemble Model...")
improved_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
improved_model.fit(X_train, y_train)

# Calculate Improved PR-AUC success measure
y_prob_imp = improved_model.predict_proba(X_test)[:, 1]
precision_i, recall_i, _ = precision_recall_curve(y_test, y_prob_imp)
improved_pr_auc = auc(recall_i, precision_i)

# --- STEP 5: DISPLAY RESULTS SIDE-BY-SIDE ---
print("\n" + "="*45)
print("             QUANTIC ASSIGNMENT RESULTS       ")
print("="*45)
print(f"BASELINE MODEL PR-AUC SCORE : {baseline_pr_auc:.4f}")
print(f"IMPROVED MODEL PR-AUC SCORE : {improved_pr_auc:.4f}")
print(f"NET PRECISION PERFORMANCE GAIN: +{(improved_pr_auc - baseline_pr_auc)*100:.2f}%")
print("="*45)

--- Establishing SQL Database Connection Pipeline ---


C:\Users\Administrator\AppData\Local\Temp\ipykernel_4532\1786691407.py:21: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query("SELECT * FROM staging_credit_risk;", conn)


Data pipeline connected! Loaded 32581 loan records.

Training Baseline Logistic Regression Model...


C:\Users\Administrator\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 5000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=5000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Training Improved Random Forest Ensemble Model...

             QUANTIC ASSIGNMENT RESULTS       
BASELINE MODEL PR-AUC SCORE : 0.6919
IMPROVED MODEL PR-AUC SCORE : 0.8722
NET PRECISION PERFORMANCE GAIN: +18.03%


In [6]:
!pip install pandas numpy psycopg2-binary scikit-learn

'pip' is not recognized as an internal or external command,
operable program or batch file.


In [8]:
import sys
!{sys.executable} -m pip install pandas numpy psycopg2-binary scikit-learn

In [3]:
# 1. Install the formatting utility directly in your notebook
import sys
!{sys.executable} -m pip install tabulate

# 2. Import the library
from tabulate import tabulate

# 3. Structure your data into a dictionary matching your real metrics
data = [
    ["Baseline Model (Logistic Regression)", "0.6919", "Linear Equation", "Weak Scaling"],
    ["Improved Model (Random Forest)", "0.8722", "100 Decision Trees", "High Stability"]
]

headers = ["Evaluation Metric", "PR-AUC Score", "Complexity", "Stability"]

# 4. Generate the table formatted as clean GitHub Markdown
print(tabulate(data, headers=headers, tablefmt="github"))



| Evaluation Metric                    |   PR-AUC Score | Complexity         | Stability      |
|--------------------------------------|----------------|--------------------|----------------|
| Baseline Model (Logistic Regression) |         0.6919 | Linear Equation    | Weak Scaling   |
| Improved Model (Random Forest)       |         0.8722 | 100 Decision Trees | High Stability |


In [7]:
import pandas as pd
import dataframe_image as dfi

metrics_data = {
    "Evaluation Metric": ["Baseline Model (Logistic Regression)", "Improved Model (Random Forest)"],
    "PR-AUC Score": [0.6919, 0.8722], 
    "Complexity": ["Linear Equation", "100 Decision Trees"],
    "Stability Status": ["Weak Scaling (Warning Issued)", "Optimal Performance"]
}
df_metrics = pd.DataFrame(metrics_data)

styled_table = (df_metrics.style
    .set_caption("PRODUCTION CREDIT RISK ENGINE: MODEL BENCHMARKING")
    # FIX: This line completely hides the 0 and 1 index columns from the exported image
    .hide(axis="index") 
    .set_properties(**{
        'background-color': '#111827',
        'color': '#F3F4F6',
        'font-family': 'Calibri, Arial',
        'border-color': '#374151',
        'text-align': 'center'
    })
    .set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#1F2937'), ('color', '#38BDF8'), ('font-weight', 'bold'), ('font-size', '14px')]},
        {'selector': 'caption', 'props': [('color', '#10B981'), ('font-weight', 'bold'), ('font-size', '16px'), ('padding', '10px')]}
    ])
    .highlight_max(subset=["PR-AUC Score"], color="#065F46", axis=0) 
    .highlight_min(subset=["PR-AUC Score"], color="#7F1D1D", axis=0)
    .format({"PR-AUC Score": "{:.4f}"}) # Change to "{:.2%}" here if you want percentages instead!
)

dfi.export(styled_table, "quantic_colorful_metrics_table.png", table_conversion="matplotlib")
print("SUCCESS: The ultimate index-free table graphic has been generated!")



SUCCESS: The ultimate index-free table graphic has been generated!
